# Transformations Test

## What is tested

This notebook checks the functions of `mars.population.transform` that change the basis of
operators and superoperators and that convert vectorized objects between the Kronecker and
tensor-product orderings.

## List of tests

**Simple tests** (`run_tests_simple`)

1. `test_hadamard_basis_change`: a density matrix transformed by the 2x2 Hadamard-like basis change gives the same result in Hilbert and Liouville space.
2. `test_single_spin_reshaping`: for one spin the Kronecker and tensor-product orderings round-trip without loss.
3. `test_reshape_flatten_consistency`: `flatten()` and `reshape()` give the same vectorization, and the reshaped result can be flattened back.

**Difficult tests** (`run_tests_difficult`)

4. `test_vectorized_operator_transformation`: for a generic 3x3 unitary, $U \rho U^\dagger$ equals the Liouville-space transformation $T |\rho\rangle$.
5. `test_hamiltonian_superoperator`: the Hamiltonian superoperator reproduces the commutator $-i[H, \rho]$.
6. `test_superoperator_basis_change`: a transformed superoperator equals the one built directly from the transformed Hamiltonian, and the dynamics do not depend on the basis.
7. `test_two_spin_vector_reshaping`: vector reshaping for a two-spin system [2, 2].
8. `test_three_subsystem_vector_reshaping`: vector reshaping for subsystems of different sizes [2, 3, 2].
9. `test_superoperator_reshaping_round_trip`: a superoperator survives the conversion Kronecker → tensor product → Kronecker.
10. `test_superoperator_action_consistency`: applying a superoperator gives the same result in both orderings.

## How to run

Run all cells. The last two cells call `run_tests_simple()` and `run_tests_difficult()`.
Each test prints one line when it passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from mars.population.transform import (
    Liouvilleator,
    basis_transformation,
    compute_liouville_basis_transformation,
    reshape_superoperator_kronecker_to_tensor_basis,
    reshape_superoperator_tensor_to_kronecker_basis,
    reshape_vectorized_kronecker_to_tensor_product,
    reshape_vectorized_tensor_product_to_kronecker,
)

# Tolerances for single-precision (complex64) round trips
VECTOR_TOL = 1e-5
SUPEROPERATOR_TOL = 1e-4

## Helper functions

In [2]:
def random_unitary(dim: int) -> torch.Tensor:
    """Return a random complex unitary matrix without special symmetries."""
    matrix = torch.randn(dim, dim, dtype=torch.complex64) + 1j * torch.randn(dim, dim, dtype=torch.complex64)
    q, r = torch.linalg.qr(matrix)
    diagonal = torch.diag(r)
    return q * (diagonal / diagonal.abs()).unsqueeze(0)


def random_hermitian(dim: int) -> torch.Tensor:
    """Return a random complex Hermitian matrix of the given size."""
    matrix = torch.randn(dim, dim, dtype=torch.complex64)
    return (matrix + matrix.conj().T) / 2


def generic_basis_change(seed: int = 42, dim: int = 3):
    """Return the Hilbert-space (U) and Liouville-space (T) transformations between two random bases."""
    torch.manual_seed(seed)
    basis_old = random_unitary(dim)
    basis_new = random_unitary(dim)
    U = basis_transformation(basis_old, basis_new)
    T = compute_liouville_basis_transformation(basis_old, basis_new)
    return U, T


def batched_kronecker(factors: list[torch.Tensor]) -> torch.Tensor:
    """Return the Kronecker product of several batched tensors, element by element along the batch."""
    batch_size = factors[0].shape[0]
    products = []
    for batch in range(batch_size):
        product = factors[0][batch]
        for factor in factors[1:]:
            product = torch.kron(product, factor[batch])
        products.append(product)
    return torch.stack(products)


def check_vector_reshaping(density_matrices: list[torch.Tensor], dims: list[int]) -> None:
    """Check the forward conversion against vec(rho_1) x vec(rho_2) x ... and the backward round trip."""
    batch_size = density_matrices[0].shape[0]
    vec_kronecker = batched_kronecker(density_matrices).reshape(batch_size, -1)
    vec_factors = [rho.reshape(batch_size, -1) for rho in density_matrices]
    expected_tensor = batched_kronecker(vec_factors)

    vec_tensor = reshape_vectorized_kronecker_to_tensor_product(vec_kronecker, dims)
    forward_error = (vec_tensor - expected_tensor).abs().max()
    assert forward_error < VECTOR_TOL, f"Forward conversion error for dims {dims}: {forward_error:.2e}"

    restored = reshape_vectorized_tensor_product_to_kronecker(vec_tensor, dims)
    backward_error = (vec_kronecker - restored).abs().max()
    assert backward_error < VECTOR_TOL, f"Round-trip error for dims {dims}: {backward_error:.2e}"

## Simple tests

In [3]:
def test_hadamard_basis_change():
    """Check that a Hadamard-like basis change agrees between Hilbert and Liouville space."""
    basis_old = torch.eye(2, dtype=torch.complex64)
    basis_new = torch.tensor([[1.0, 1.0], [1.0, -1.0]], dtype=torch.complex64) / torch.sqrt(torch.tensor(2.0))

    U = basis_transformation(basis_old, basis_new)
    T = compute_liouville_basis_transformation(basis_old, basis_new)

    rho_old = torch.tensor([[0.6, 0.1 + 0.2j], [0.1 - 0.2j, 0.4]], dtype=torch.complex64)
    rho_hilbert = U @ rho_old @ U.conj().T
    rho_liouville = (T @ rho_old.flatten()).reshape(2, 2)

    assert torch.allclose(rho_liouville, rho_hilbert), "Hilbert and Liouville transformations disagree"
    print("Hadamard basis change test passed (Hilbert and Liouville results agree)")


def test_single_spin_reshaping():
    """Check that vector reshaping round-trips for a single spin, where it must change nothing."""
    torch.manual_seed(42)
    batch_size, dims = 3, [4]
    vec_rho = torch.randn(batch_size, 4, 4, dtype=torch.complex64).reshape(batch_size, -1)

    vec_tensor = reshape_vectorized_kronecker_to_tensor_product(vec_rho, dims)
    restored = reshape_vectorized_tensor_product_to_kronecker(vec_tensor, dims)

    error = (vec_rho - restored).abs().max()
    assert error < VECTOR_TOL, f"Round-trip error: {error:.2e}"
    print("Single spin reshaping test passed (round trip for dims [4])")


def test_reshape_flatten_consistency():
    """Check that flatten() matches reshape() for batched density matrices and converted vectors."""
    torch.manual_seed(42)
    batch_size, dims = 4, [2, 3]
    rho_1 = torch.randn(batch_size, 2, 2)
    rho_2 = torch.randn(batch_size, 3, 3)
    rho_kronecker = batched_kronecker([rho_1, rho_2])

    for rho in (rho_1, rho_2, rho_kronecker):
        assert torch.allclose(rho.flatten(start_dim=1), rho.reshape(batch_size, -1)), (
            "flatten() and reshape() disagree"
        )

    vec_tensor = reshape_vectorized_kronecker_to_tensor_product(rho_kronecker.flatten(start_dim=1), dims)
    flattened = vec_tensor.reshape(batch_size, 4, 9).flatten(start_dim=1)
    assert torch.allclose(vec_tensor, flattened), "The converted vector cannot be reshaped and flattened back"
    print("Reshape and flatten test passed (both vectorizations agree for dims [2, 3])")

## Difficult tests

### Generic basis change

In [4]:
def test_vectorized_operator_transformation():
    """Check that U rho U^dagger equals the Liouville-space transformation for a generic 3x3 unitary."""
    U, T = generic_basis_change()
    rho_old = random_hermitian(3)

    rho_hilbert = U @ rho_old @ U.conj().T
    rho_liouville = (T @ rho_old.flatten()).reshape(3, 3)

    assert torch.allclose(rho_hilbert, rho_liouville, atol=1e-6), "Hilbert and Liouville transformations disagree"
    print("Vectorized operator test passed (U rho U^dagger equals T |rho> for a generic unitary)")


def test_hamiltonian_superoperator():
    """Check that the Hamiltonian superoperator reproduces the commutator -i[H, rho]."""
    torch.manual_seed(42)
    rho = random_hermitian(3)
    hamiltonian = random_hermitian(3)

    commutator = -1j * (hamiltonian @ rho - rho @ hamiltonian)
    superop = Liouvilleator().hamiltonian_superop(hamiltonian)
    commutator_liouville = (superop @ rho.flatten()).reshape(3, 3)

    assert torch.allclose(commutator, commutator_liouville, atol=1e-6), "The superoperator gives a wrong commutator"
    print("Hamiltonian superoperator test passed (it reproduces -i[H, rho])")


def test_superoperator_basis_change():
    """Check the similarity transform of a superoperator and the basis independence of the dynamics."""
    U, T = generic_basis_change()
    liouvilleator = Liouvilleator()
    rho_old = random_hermitian(3)
    hamiltonian_old = random_hermitian(3)
    hamiltonian_new = U @ hamiltonian_old @ U.conj().T

    superop_transformed = T @ liouvilleator.hamiltonian_superop(hamiltonian_old) @ T.conj().transpose(-2, -1)
    superop_direct = liouvilleator.hamiltonian_superop(hamiltonian_new)
    assert torch.allclose(superop_transformed, superop_direct, atol=1e-6), (
        "The transformed superoperator differs from the directly built one"
    )

    rho_new = U @ rho_old @ U.conj().T
    derivative_direct = -1j * (hamiltonian_new @ rho_new - rho_new @ hamiltonian_new)
    derivative_transformed = (superop_transformed @ (T @ rho_old.flatten())).reshape(3, 3)
    assert torch.allclose(derivative_direct, derivative_transformed, atol=1e-6), (
        "The dynamics depend on the basis"
    )
    print("Superoperator basis change test passed (similarity transform and basis-independent dynamics)")

### Kronecker and tensor-product orderings

In [5]:
def test_two_spin_vector_reshaping():
    """Check the vector conversion for a two-spin system with dims [2, 2]."""
    torch.manual_seed(42)
    batch_size = 3
    rho_1 = torch.randn(batch_size, 2, 2, dtype=torch.complex64)
    rho_2 = torch.randn(batch_size, 2, 2, dtype=torch.complex64)
    rho_1 = (rho_1 + rho_1.conj().transpose(-1, -2)) / 2
    rho_2 = (rho_2 + rho_2.conj().transpose(-1, -2)) / 2

    check_vector_reshaping([rho_1, rho_2], dims=[2, 2])
    print("Two-spin reshaping test passed (forward conversion and round trip for dims [2, 2])")


def test_three_subsystem_vector_reshaping():
    """Check the vector conversion for three subsystems of different sizes, dims [2, 3, 2]."""
    torch.manual_seed(42)
    batch_size = 2
    rho_1 = torch.randn(batch_size, 2, 2, dtype=torch.complex64)
    rho_2 = torch.randn(batch_size, 3, 3, dtype=torch.complex64)
    rho_3 = torch.randn(batch_size, 2, 2, dtype=torch.complex64)

    check_vector_reshaping([rho_1, rho_2, rho_3], dims=[2, 3, 2])
    print("Three-subsystem reshaping test passed (forward conversion and round trip for dims [2, 3, 2])")


def test_superoperator_reshaping_round_trip():
    """Check that a superoperator is unchanged by Kronecker -> tensor product -> Kronecker."""
    torch.manual_seed(42)
    dims = [2, 2]
    superop_kronecker = torch.randn(2, 16, 16, dtype=torch.complex64)

    superop_tensor = reshape_superoperator_kronecker_to_tensor_basis(superop_kronecker, dims)
    restored = reshape_superoperator_tensor_to_kronecker_basis(superop_tensor, dims)

    error = (superop_kronecker - restored).abs().max()
    assert error < SUPEROPERATOR_TOL, f"Round-trip error: {error:.2e}"
    print("Superoperator round-trip test passed (Kronecker -> tensor product -> Kronecker)")


def test_superoperator_action_consistency():
    """Check that a superoperator acting on a vectorized state agrees in both orderings."""
    torch.manual_seed(42)
    batch_size, dims = 2, [2, 2]
    rho = torch.randn(batch_size, 4, 4, dtype=torch.complex64)
    rho = (rho + rho.conj().transpose(-1, -2)) / 2
    vec_kronecker = rho.reshape(batch_size, -1)
    superop_kronecker = torch.randn(batch_size, 16, 16, dtype=torch.complex64)

    result_kronecker = torch.bmm(superop_kronecker, vec_kronecker.unsqueeze(-1)).squeeze(-1)

    vec_tensor = reshape_vectorized_kronecker_to_tensor_product(vec_kronecker, dims)
    superop_tensor = reshape_superoperator_kronecker_to_tensor_basis(superop_kronecker, dims)
    result_tensor = torch.bmm(superop_tensor, vec_tensor.unsqueeze(-1)).squeeze(-1)
    result_converted = reshape_vectorized_tensor_product_to_kronecker(result_tensor, dims)

    error = (result_kronecker - result_converted).abs().max()
    assert error < SUPEROPERATOR_TOL, f"Results differ between orderings: {error:.2e}"
    print("Superoperator action test passed (same result in both orderings)")

## Run the tests

In [6]:
def run_tests_simple():
    """Run the simple tests of this notebook."""
    tests = [test_hadamard_basis_change, test_single_spin_reshaping, test_reshape_flatten_consistency]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} simple transformation tests passed.")


def run_tests_difficult():
    """Run the difficult tests of this notebook."""
    tests = [
        test_vectorized_operator_transformation,
        test_hamiltonian_superoperator,
        test_superoperator_basis_change,
        test_two_spin_vector_reshaping,
        test_three_subsystem_vector_reshaping,
        test_superoperator_reshaping_round_trip,
        test_superoperator_action_consistency,
    ]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} difficult transformation tests passed.")

In [7]:
run_tests_simple()

Hadamard basis change test passed (Hilbert and Liouville results agree)
Single spin reshaping test passed (round trip for dims [4])
Reshape and flatten test passed (both vectorizations agree for dims [2, 3])

All 3 simple transformation tests passed.


In [8]:
run_tests_difficult()

Vectorized operator test passed (U rho U^dagger equals T |rho> for a generic unitary)
Hamiltonian superoperator test passed (it reproduces -i[H, rho])
Superoperator basis change test passed (similarity transform and basis-independent dynamics)
Two-spin reshaping test passed (forward conversion and round trip for dims [2, 2])
Three-subsystem reshaping test passed (forward conversion and round trip for dims [2, 3, 2])
Superoperator round-trip test passed (Kronecker -> tensor product -> Kronecker)
Superoperator action test passed (same result in both orderings)

All 7 difficult transformation tests passed.
